In [1]:
from pyspark.sql import SparkSession
from pyspark.streaming import StreamingContext
import os
from pyspark.sql.functions import max,min
import logging

In [2]:
# definaion of the variable for kafka
topic_name = "debezium.commerce.products"
bootstrap_server = "localhost:9093"
spark_version= "3.5.2"
scala_version = "2.12"

In [3]:
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version},org.apache.hadoop:hadoop-aws:3.3.1,com.amazonaws:aws-java-sdk-bundle:1.11.874 pyspark-shell'
spark = SparkSession.builder\
   .master("local")\
   .config("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.5.2,com.amazonaws:aws-java-sdk-bundle:1.11.874") \
   .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
   .config("spark.hadoop.fs.s3a.access.key", "minio") \
   .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
   .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
   .config("spark.hadoop.fs.s3a.path.style.access", "true") \
   .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")\
   .appName("kafka-streaming")\
   .getOrCreate()
# set logger level with format
logging.basicConfig(format='%(asctime)s %(levelname)-8s %(message)s', level=logging.INFO, datefmt='%Y-%m-%d %H:%M:%S')

24/09/22 15:15:48 WARN Utils: Your hostname, depq-2.local resolves to a loopback address: 127.0.0.1; using 192.168.1.205 instead (on interface en0)
24/09/22 15:15:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/lap14646/Library/Python/3.9/lib/python/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/lap14646/.ivy2/cache
The jars for the packages stored in: /Users/lap14646/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7344924f-24d1-4d5b-82eb-a889d1796b25;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.2 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in local-m2-cache
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in local-m2-cache
	found org.apache.hadoop#hadoop-client-api;3.3.4 in local-m2-cache
	found commons-logging#commons-logging;1.1.3 in local-m2-cache
	found com.google.code.findbugs#jsr

In [4]:
spark

In [5]:
def read_df_from_kafka(topic_name, bootstrap_server):
    try:
        df_cdc = spark.readStream.format("kafka") \
            .option("kafka.bootstrap.servers", bootstrap_server) \
            .option("subscribe", topic_name) \
            .option("startingOffsets", "earliest") \
            .option("kafka.security.protocol", "PLAINTEXT") \
            .load()
        logging.info("Reading from Kafka successfully!")
    except Exception as e:
        logging.error(f"Error reading from Kafka: {e}")
    return df_cdc

In [12]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
# create structype for schema
product_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("price", FloatType(), True)
])
data_schema = StructType([
        StructField("payload", StructType([
            StructField("after", product_schema, True)
        ]), True)
    ])

In [13]:
data_schema

StructType([StructField('payload', StructType([StructField('after', StructType([StructField('id', IntegerType(), True), StructField('name', StringType(), True), StructField('description', StringType(), True), StructField('price', FloatType(), True)]), True)]), True)])

In [14]:
# Extract value from the Kafka DataFrame (Kafka messages are typically serialized as bytes)
df_cdc = read_df_from_kafka(topic_name, bootstrap_server)

2024-09-22 15:19:47 INFO     Reading from Kafka successfully!


In [16]:
# transformation cdc debezium format
parsed_df = df_cdc.selectExpr("CAST(value AS STRING) as json") \
.select(from_json(col("json"), data_schema).alias("data")) \
.select("data.payload.after.*")

In [20]:
# how to write to s3 minio
parsed_df.writeStream.format("parquet") \
    .option("path", "s3a://raw/") \
    .option("checkpointLocation", "s3a://raw/checkpoint") \
    .outputMode("append") \
    .trigger(processingTime="60 seconds") \
    .start() \
    .awaitTermination()

24/09/22 15:27:42 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
24/09/22 15:27:42 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
24/09/22 15:36:00 ERROR FileFormatWriter: Aborting job f89ec1e7-d36d-4502-acc9-01eeac69f480.
org.apache.spark.SparkFileNotFoundException: [BATCH_METADATA_NOT_FOUND] Unable to find batch s3a://raw/_spark_metadata/0.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.batchMetadataFileNotFoundError(QueryExecutionErrors.scala:1849)
	at org.apache.spark.sql.execution.streaming.HDFSMetadataLog.applyFnToBatchByStream(HDFSMetadataLog.scala:194)
	at org.apache.spark.sql.execution.streaming.CompactibleFileStreamLog.applyFnInBatch(CompactibleFileStreamLog.scala:207)
	at org.apache.spark.sql.execution.streaming.CompactibleFileStreamLog.foreachInBatch(Compactib

StreamingQueryException: [STREAM_FAILED] Query [id = f607cf4b-7971-42c3-b938-e100c38ff563, runId = d3b71f78-0147-43b1-bc85-ac449d4fdc4c] terminated with exception: [BATCH_METADATA_NOT_FOUND] Unable to find batch s3a://raw/_spark_metadata/0.